In [1]:
!python --version  

Python 3.13.12


In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

# avoid warnings for old claude model use
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
# Initialize Antropic Chat client
from anthropic import Anthropic
client = Anthropic()

In [5]:
# Helper Chat Functions


## Function to Append User messasge
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


## Function to Append Assistant messasge
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


## Main Function for chat
def get_response(messages, stop_sequences=None, system=None):

    model = "claude-sonnet-4-0"  # you can change the model according to your usecase
    params = {
        "model": model,
        "max_tokens": 1500,
        "messages": messages,
    }
    if system:
        params["system"] = system    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message


def get_chat_content(message):
    return message.content[0].text

In [6]:
def getStructureOutput(mesg, data_prefix="```json", stop_sequences=None):
    messages = []

    # User's request > what user wants in specific format
    add_user_message(messages, mesg)

    # start the content and make the AI generate the remaining part.
    add_assistant_message(messages, data_prefix)

    response = get_response(messages=messages, stop_sequences=stop_sequences)
    return get_chat_content(response)

In [ ]:
# With less temprature we will likely get the almost same responses each time
import json

response = getStructureOutput(
    "Generate a very short event bridge rule as json",
    stop_sequences=["```"],
)
# Used strip to remove \n and white spaces
# json.loads(response.strip())

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [ ]:
# Formatted json preview
print(f"{response.strip()}")

{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["myapp.orders"],
    "detail-type": ["Order Placed"]
  },
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ]
}
